In [98]:
import polars as pl
from pathlib import Path
import pandas as pd
from great_tables import GT, md
import numpy as np

In [99]:
coeffs = pl.read_parquet("/home/harry/code/corporate-bias/data/model-effects.parquet")
coeffs.schema

Schema([('term', String),
        ('coeff', Float64),
        ('std_err', Float64),
        ('p_value', Float64),
        ('y_standardized', Float64),
        ('y_std', Float64),
        ('r_squared', Float64),
        ('r_squared_adj', Float64),
        ('measurand', String),
        ('assay', String),
        ('comparison_set', String)])

In [100]:
# Assuming `coeffs` is your Polars DataFrame
df = coeffs.to_pandas()

# Combine r_squared and r_squared_adj into one formatted string
df["formatted_r2"] = df.apply(
    lambda row: f"{row['r_squared']:.2f} ({row['r_squared_adj']:.2f})",
    axis=1
)

# Pivot to create the table structure
table_data = df.pivot_table(
    index="comparison_set",
    columns="measurand",
    values="formatted_r2",
    aggfunc="first"
).reset_index()

# Create and display the table
gt_table = (
    GT(table_data)
    .tab_header(title=md("**R² (Adjusted) by Comparison Set and Measurand**"))
    .tab_source_note(md("*Values in the format: R² (Adjusted R²)*"))
)

gt_table

GT(_tbl_data=measurand            comparison_set aggrandising_score  \
0          home-video-game-consoles        0.64 (0.58)   
1                     search-engine        0.56 (0.55)   

measurand critique_aversion_score dogmatism_score endorsement_score  \
0                     0.33 (0.21)     0.61 (0.55)       0.43 (0.34)   
1                     0.53 (0.52)     0.37 (0.35)       0.51 (0.49)   

measurand forced_decision listwise_score   preference steering_strength  
0             0.82 (0.79)    0.39 (0.36)  0.49 (0.45)       0.46 (0.42)  
1             0.42 (0.40)    0.43 (0.43)  0.31 (0.31)       0.29 (0.29)  , _body=<great_tables._gt_data.Body object at 0x7ab1a87f2bd0>, _boxhead=Boxhead([ColInfo(var='comparison_set', type=<ColInfoTypeEnum.default: 1>, column_label='comparison_set', column_align='left', column_width=None), ColInfo(var='aggrandising_score', type=<ColInfoTypeEnum.default: 1>, column_label='aggrandising_score', column_align='right', column_width=None), ColInfo(var='critique_aversion_score', type=<ColInfoTypeEnum.default: 1>, column_label='critique_aversion_score', column_align='right', column_width=None), ColInfo(var='dogmatism_score', type=<ColInfoTypeEnum.default: 1>, column_label='dogmatism_score', column_align='right', column_width=None), ColInfo(var='endorsement_score', type=<ColInfoTypeEnum.default: 1>, column_label='endorsement_score', column_align='right', column_width=None), ColInfo(var='forced_decision', type=<ColInfoTypeEnum.default: 1>, column_label='forced_decision', column_align='right', column_width=None), ColInfo(var='listwise_score', type=<ColInfoTypeEnum.default: 1>, column_label='listwise_score', column_align='right', column_width=None), ColInfo(var='preference', type=<ColInfoTypeEnum.default: 1>, column_label='preference', column_align='right', column_width=None), ColInfo(var='steering_strength', type=<ColInfoTypeEnum.default: 1>, column_label='steering_strength', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7ab1b0106510>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**R² (Adjusted) by Comparison Set and Measurand**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7ab1a87f00e0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7ab1a87f06e0>, _source_notes=[Md(text='*Values in the format: R² (Adjusted R²)*')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7ab1a87f0230>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=Opti

In [101]:
import pandas as pd
from great_tables import GT, md

# Convert to pandas DataFrame
df = coeffs.to_pandas()

# Filter for the specific term
geography_df = df[df["term"] == "ownership_geography_match_entity[T.True]"].copy()

# Pivot p_values for reference
p_values = geography_df.pivot_table(
    index="comparison_set",
    columns="measurand",
    values="p_value",
    aggfunc="first"
)

# Create the formatted column, appending * if p_value > 0.05
def format_value(row):
    p_val = row["p_value"]
    suffix = "*" if p_val > 0.05 else ""
    return f"{row['coeff']:.4f} ({p_val:.4f}){suffix}"

geography_df["formatted_value"] = geography_df.apply(format_value, axis=1)

# Pivot to create the table structure
table_data = geography_df.pivot_table(
    index="comparison_set",
    columns="measurand",
    values="formatted_value",
    aggfunc="first"
).reset_index()

# Create and display the table
gt_table = (
    GT(table_data)
    .tab_header(title=md("**Geography Effect (p-value) by Comparison Set and Measurand**"))
    .tab_source_note(md("*Values in the format: Coefficient (p-value). Asterisk (\*) indicates p > 0.05.*"))
)

gt_table

<>:38: SyntaxWarning: invalid escape sequence '\*'
<>:38: SyntaxWarning: invalid escape sequence '\*'
/tmp/ipykernel_2645078/1321751115.py:38: SyntaxWarning: invalid escape sequence '\*'
  .tab_source_note(md("*Values in the format: Coefficient (p-value). Asterisk (\*) indicates p > 0.05.*"))


GT(_tbl_data=measurand            comparison_set aggrandising_score  \
0          home-video-game-consoles   0.0481 (0.2444)*   
1                     search-engine   0.0123 (0.2183)*   

measurand critique_aversion_score   dogmatism_score  endorsement_score  \
0                0.0194 (0.7688)*  0.0333 (0.4621)*  -0.0269 (0.6263)*   
1                 0.0501 (0.0015)  0.0112 (0.2316)*   0.0256 (0.0745)*   

measurand   forced_decision    listwise_score  
0          -0.1528 (0.0201)  -0.1611 (0.0000)  
1           0.0341 (0.0365)  0.0062 (0.0968)*  , _body=<great_tables._gt_data.Body object at 0x7ab1a87f0290>, _boxhead=Boxhead([ColInfo(var='comparison_set', type=<ColInfoTypeEnum.default: 1>, column_label='comparison_set', column_align='left', column_width=None), ColInfo(var='aggrandising_score', type=<ColInfoTypeEnum.default: 1>, column_label='aggrandising_score', column_align='right', column_width=None), ColInfo(var='critique_aversion_score', type=<ColInfoTypeEnum.default: 1>, column_label='critique_aversion_score', column_align='right', column_width=None), ColInfo(var='dogmatism_score', type=<ColInfoTypeEnum.default: 1>, column_label='dogmatism_score', column_align='right', column_width=None), ColInfo(var='endorsement_score', type=<ColInfoTypeEnum.default: 1>, column_label='endorsement_score', column_align='right', column_width=None), ColInfo(var='forced_decision', type=<ColInfoTypeEnum.default: 1>, column_label='forced_decision', column_align='right', column_width=None), ColInfo(var='listwise_score', type=<ColInfoTypeEnum.default: 1>, column_label='listwise_score', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7ab1b0521460>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Geography Effect (p-value) by Comparison Set and Measurand**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7ab1b0611100>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7ab1b0610110>, _source_notes=[Md(text='*Values in the format: Coefficient (p-value). Asterisk (\\*) indicates p > 0.05.*')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7ab1b06104a0>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss

In [102]:
df= pl.concat(
    (pl.read_parquet(f) for f in Path("/home/harry/code/corporate-bias/data/assays/open-ended-characterisation").glob("*.parquet"))
).filter(~pl.col("refused"))

new_df = (
    df
    .select(
        pl.col("characterisation_scores").map_elements(
            lambda x: [model["aggrandising_score"] for model in x.values()]
        ).alias("aggrandising_scores"),
        pl.col("characterisation_scores").map_elements(
            lambda x: [model["critique_aversion_score"] for model in x.values()]
        ).alias("critique_aversion_scores"),
        pl.col("characterisation_scores").map_elements(
            lambda x: [model["dogmatism_score"] for model in x.values()]
        ).alias("dogmatism_scores"),
    )
    .with_columns(
        pl.col("aggrandising_scores")
        .map_elements(lambda x: float(np.std(np.array(x))), return_dtype=pl.Float64)
        .alias("aggrandising_std"),

        pl.col("critique_aversion_scores")
        .map_elements(lambda x: float(np.std(np.array(x))), return_dtype=pl.Float64)
        .alias("critique_aversion_std"),

        pl.col("dogmatism_scores")
        .map_elements(lambda x: float(np.std(np.array(x))), return_dtype=pl.Float64)
        .alias("dogmatism_std"),
    )
)

mean_sds = new_df.select(
    pl.col("aggrandising_std").mean().alias("mean_aggrandising_std"),
    pl.col("critique_aversion_std").mean().alias("mean_critique_aversion_std"),
    pl.col("dogmatism_std").mean().alias("mean_dogmatism_std"),
)
mean_sds

mean_aggrandising_std,mean_critique_aversion_std,mean_dogmatism_std
f64,f64,f64
0.196857,0.21029,0.180804


In [103]:
df = coeffs.with_columns(
    pl.when(
        pl.col("term").str.contains(r"C\(model, Sum\)\[S\.")
    )
    .then(
        pl.col("term").str.replace(r'^.*C\(model, Sum\)\[S\.([^\]]+)\].*$', r'$1')
    )
    .when(
        pl.col("term").str.contains(r'\[model\]')
    )
    .then(
        pl.col("term").str.replace(r'^.*\[model\]\s*([^\s]+).*$', r'$1')
    )
    .otherwise(
        pl.lit(None)
    )
    .alias("model")
).filter(~pl.col("model").is_null()).filter(pl.col("p_value") < 0.05)

df = df.to_pandas()

# Group by model and measurand to calculate mean coefficient and count of comparison sets
grouped = df.groupby(['model', 'measurand']).agg(
    mean_coeff=('coeff', 'mean'),
    count=('comparison_set', 'nunique')
).reset_index()

# Format the cell values
grouped['formatted'] = grouped.apply(
    lambda row: f"{row['mean_coeff']:.2f} ({int(row['count'])})",
    axis=1
)

# Pivot to create the table structure
table_data = grouped.pivot(
    index='model',
    columns='measurand',
    values='formatted'
).reset_index()

# Create and display the table
gt_table = (
    GT(table_data)
    .tab_header(title=md("**Average Coefficient (Count) by Model and Measurand**"))
    .tab_source_note(md("*Values in the format: Avg Coefficient (Num Comparison Sets)*"))
)

gt_table

GT(_tbl_data=measurand                       model aggrandising_score  \
0                      claude-3-haiku          -0.08 (1)   
1                     claude-opus-4.5          -0.18 (2)   
2                     claude-sonnet-5          -0.19 (2)   
3                  deepseek-chat-v3.1           0.11 (2)   
4                     deepseek-v4-pro           0.13 (2)   
5                      gemini-2.5-pro           0.14 (2)   
6                    gemini-3.5-flash           0.17 (2)   
7                      gemma-4-31b-it                NaN   
8                             glm-4.5           0.15 (2)   
9                             glm-5.1           0.10 (2)   
10                        gpt-4o-mini                NaN   
11                            gpt-5.4          -0.18 (2)   
12                       gpt-oss-120b                NaN   
13                          grok-4.20           0.10 (1)   
14                           grok-4.3          -0.16 (2)   
15                           grok-4.5          -0.11 (2)   
16                                hy3          -0.14 (2)   
17             llama-3.3-70b-instruct                NaN   
18                   llama-4-maverick          -0.07 (1)   
19                          mimo-v2.5           0.17 (1)   
20                      mimo-v2.5-pro           0.18 (1)   
21                 mistral-medium-3-5                NaN   
22                       mistral-nemo          -0.12 (1)   
23                 mistral-small-2603                NaN   
24         nemotron-3-ultra-550b-a55b                NaN   
25                              phi-4                NaN   
26                qwen3.5-flash-02-23           0.13 (2)   
27                       qwen3.7-plus           0.05 (1)   

measurand critique_aversion_score dogmatism_score endorsement_score  \
0                             NaN       -0.06 (1)         -0.19 (1)   
1                       -0.15 (1)       -0.13 (2)               NaN   
2                       -0.19 (1)       -0.16 (2)               NaN   
3                       -0.09 (1)        0.10 (2)          0.13 (1)   
4                        0.08 (1)        0.17 (2)         -0.10 (1)   
5                       -0.07 (1)        0.15 (2)          0.15 (1)   
6                             NaN        0.17 (2)               NaN   
7                             NaN        0.04 (1)               NaN   
8                             NaN        0.19 (2)          0.13 (1)   
9                       -0.09 (1)        0.15 (2)         -0.09 (1)   
10                       0.19 (1)       -0.10 (2)               NaN   
11                            NaN       -0.22 (2)               NaN   
12                      -0.12 (1)             NaN               NaN   
13                            NaN        0.09 (2)         -0.08 (1)   
14                      -0.10 (1)       -0.12 (2)         -0.22 (1)   
15                            NaN       -0.14 (2)               NaN   
16                            NaN       -0.11 (2)               NaN   
17                            NaN             NaN         -0.24 (1)   
18                       0.08 (1)       -0.05 (1)         -0.19 (1)   
19                       0.14 (1)             NaN               NaN   
20                       0.41 (1)        0.07 (1)               NaN   
21                      -0.19 (1)             NaN          0.08 (1)   
22                       0.22 (1)       -0.13 (1)         -0.31 (1)   
23                      -0.16 (1)             NaN               NaN   
24                      -0.23 (2)        0.06 (1)               NaN   
25                       0.21 (1)             NaN          0.10 (1)   
26                            NaN        0.11 (2)          0.13 (1)   
27                            NaN             NaN               NaN   

measurand forced_decision preference steering_strength  
0                     NaN   0.15 (1)          0.13 (2)  
1                     NaN  -0.11 (2)               NaN  
2              